In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import RandomOverSampler

import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd

# Load original dataset
df = pd.read_csv("fraudTrain.csv")

# Create smaller sample (adjust rows if needed)
sample_df = df.sample(n=90000, random_state=42)

# Save compressed version
sample_df.to_csv(
    "fraudTrain_sample.csv",
    index=False,
    float_format='%.2f'
)

# Check file size
import os

size_mb = os.path.getsize("fraudTrain_sample.csv") / (1024 * 1024)

print(f"File saved successfully!")
print(f"Rows: {len(sample_df)}")
print(f"Columns: {len(sample_df.columns)}")
print(f"Size: {size_mb:.2f} MB")

File saved successfully!
Rows: 90000
Columns: 23
Size: 21.88 MB


In [3]:
df = pd.read_csv("fraudTrain.csv")

print(df.shape)
df.head()

(1296675, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  str    
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  str    
 4   category               1296675 non-null  str    
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  str    
 7   last                   1296675 non-null  str    
 8   gender                 1296675 non-null  str    
 9   street                 1296675 non-null  str    
 10  city                   1296675 non-null  str    
 11  state                  1296675 non-null  str    
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long                   129667

In [5]:
df.isnull().sum()

Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

In [6]:
df['is_fraud'].value_counts()

is_fraud
0    1289169
1       7506
Name: count, dtype: int64

In [7]:
df['is_fraud'].value_counts(normalize=True)*100

is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64

In [8]:
drop_cols = [
    'Unnamed: 0',
    'trans_date_trans_time',
    'cc_num',
    'first',
    'last',
    'street',
    'trans_num'
]

df = df.drop(columns=drop_cols)

In [9]:
df['dob'] = pd.to_datetime(df['dob'])

current_year = 2026

df['age'] = current_year - df['dob'].dt.year

df.drop('dob', axis=1, inplace=True)

In [10]:
categorical_cols = df.select_dtypes(include='object').columns

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

df.head()

,merchant,category,amt,gender,city,state,zip,lat,long,city_pop,job,unix_time,merch_lat,merch_long,is_fraud,age
0,514,8,4.97,0,526,27,28654,36.0788,-81.1781,3495,370,1325376018,36.011293,-82.048315,0,38
1,241,4,107.23,0,612,47,99160,48.8878,-118.2105,149,428,1325376044,49.159047,-118.186462,0,48
2,390,0,220.11,1,468,13,83252,42.1808,-112.2620,4154,307,1325376051,43.150704,-112.154481,0,64
3,360,2,45.00,1,84,26,59632,46.2306,-112.1138,1939,328,1325376076,47.034331,-112.561071,0,59
4,297,9,41.96,1,216,45,24433,38.4207,-79.4629,99,116,1325376186,38.674999,-78.632459,0,40


In [11]:
X = df.drop('is_fraud', axis=1)

y = df['is_fraud']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1037340, 15)
(259335, 15)


In [13]:
ros = RandomOverSampler(random_state=42)

X_train_resampled, y_train_resampled = ros.fit_resample(
    X_train,
    y_train
)

print("Before Oversampling:")
print(y_train.value_counts())

print("\nAfter Oversampling:")
print(y_train_resampled.value_counts())

Before Oversampling:
is_fraud
0    1031335
1       6005
Name: count, dtype: int64

After Oversampling:
is_fraud
0    1031335
1    1031335
Name: count, dtype: int64


In [14]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = "N/A"

    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("ROC-AUC  :", auc)

    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report")
    print(classification_report(y_test, y_pred))

In [15]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_resampled, y_train_resampled)

print("LOGISTIC REGRESSION RESULTS")
evaluate_model(lr, X_test, y_test)

LOGISTIC REGRESSION RESULTS
Accuracy : 0.9517226753041433
Precision: 0.0859075535512965
Recall   : 0.7614923384410394
F1 Score : 0.15439686613535053
ROC-AUC  : 0.8354556810969334

Confusion Matrix
[[245672  12162]
 [   358   1143]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.95      0.98    257834
           1       0.09      0.76      0.15      1501

    accuracy                           0.95    259335
   macro avg       0.54      0.86      0.56    259335
weighted avg       0.99      0.95      0.97    259335



In [16]:
dt = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

dt.fit(X_train_resampled, y_train_resampled)

print("DECISION TREE RESULTS")
evaluate_model(dt, X_test, y_test)

DECISION TREE RESULTS
Accuracy : 0.9624963849846724
Precision: 0.12927071125935274
Recall   : 0.955363091272485
F1 Score : 0.22772748928060982
ROC-AUC  : 0.9825665930406127

Confusion Matrix
[[248175   9659]
 [    67   1434]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    257834
           1       0.13      0.96      0.23      1501

    accuracy                           0.96    259335
   macro avg       0.56      0.96      0.60    259335
weighted avg       0.99      0.96      0.98    259335



In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_resampled, y_train_resampled)

print("RANDOM FOREST RESULTS")
evaluate_model(rf, X_test, y_test)

In [ ]:
models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf
}

results = []

for name, model in models.items():

    y_pred = model.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ])

comparison = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

comparison.sort_values(
    by="F1 Score",
    ascending=False
)

In [ ]:
sample = X_test.iloc[[0]]

prediction = rf.predict(sample)

if prediction[0] == 1:
    print("Fraudulent Transaction")
else:
    print("Legitimate Transaction")